# StarLayer User Guide (Notebook)

This notebook is located at https://github.com/hidden-graph/starlayer/blob/main/docs/user-guide-v1.ipynb

## How to run this notebook

1. pip install from the github repository.
2. Run cells from top to bottom so shared variables remain available.


In [1]:
#pip install "git+https://github.com/hidden-graph/starlayer.git"

In [2]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics
Extension of the rdflib graph model to support RDF 1.2.  
- Triple terms and statement resources
- Reification via rdf:reifies and statement metadata
- Direction-tagged strings such as "hello"@en--ltr and "مرحبا"@ar--rtl

In [3]:
#create the graph, and assign namespace
g = StarLayerGraph()
g.bind("ex", EX)

#create a triple term
tt = TripleTerm(EX.bob, EX.knows, EX.carol)

#create a reifer (EX.clain) associated with triple term and add to graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.wikipedia))

print((EX.claim, RDF.reifies, tt) in g)
print(g.serialize(format="turtle12"))

True
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



In [4]:
g = StarLayerGraph()
g.bind("ex", EX)

#add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.other, RDF.reifies, (EX.bob, EX.likes, EX.dana)))

g.add((EX.bob,EX.knows,EX.dana))

#rdflib triples now accepts triple term as the object when selecting triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.bob):
    print(t)
#has_triple_term tests whether the triple term is in the graph.
#the triple term X.bob, EX.knows, EX.dana is asserted in the graph, but is not the 
#object of a triple and so returns false
print(g.has_triple_term(EX.bob, EX.knows, EX.carol))
print(g.has_triple_term(EX.bob, EX.knows, EX.dana))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
True
False


In [5]:
#RDF.reifies is the common approach to making a statement about a statement, 
#RDF 1.2 allows triple terms in object position of any triple.

#add triple to the graph with triple term as object
g.add((EX.dana, EX.said, (EX.bob, EX.knows, EX.carol)))

#triples accepts triple term as object to select triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

#qname_term is a starlayer function that adds qname transformation to triple terms as well.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))


ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
ex:dana ex:said <<( ex:bob ex:knows ex:carol )>>


### Finding what's been said about a statement
`reifiers()`, `reifications()`, `reifier_annotations()`, `reified_triples()`, and `remove_reification()` navigate the reifier, riple-term, annotation relationships directly, without SPARQL queries. `remove_reification(reifier, triple_term=None)` can be scoped to one specific reifier↔triple link, leaving any other triple(s) the same reifier reifies — and all its annotations — untouched; omit `triple_term` for the original all-or-nothing behavior.

In [6]:
# continues using g from the previous cell

#adds assertions to the reification EX.claim
g.add((EX.claim, EX.source, EX.wikipedia))

tt1 = (EX.bob, EX.knows, EX.carol)
tt2 = (EX.bob, EX.likes, EX.dana)

# reifiers(): returns the reifier node(s) that reify a given triple term.
print([g.qname(r) for r in g.reifiers(TT=tt1)])

# reifications(): returns a list of triple terms that have at least one reifier.
for tt in g.reifications():
    print(tt)

# reifier_annotations(): returns a reifier's annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname(reifier), g.qname(pred), g.qname(val))

# reified_triples(): returns the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(tt)

# give EX.claim a second rdf:reifies link, so scoped vs. wildcard removal are distinguishable
g.add((EX.claim, RDF.reifies, tt2))

# remove_reification(reifier, triple_term): remove the reification link between a reifier and the 
# specified triple.
g.remove_reification(EX.claim, tt1)
print((EX.claim, RDF.reifies, tt1) in g)         # False - only this link removed
print((EX.claim, RDF.reifies, tt2) in g)         # True  - untouched
print((EX.claim, EX.source, EX.wikipedia) in g)  # True  - untouched

# remove_reification(reifier): removes all rdf:reifies links from reifier
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt2) in g)
print((EX.claim, EX.source, EX.wikipedia) in g)

['ex:claim']
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
ex:claim ex:source ex:wikipedia
<<( ex:bob ex:knows ex:carol )>>
False
True
True
False
True


### Set the direction of a string literal.

DirLangString has been added to set the direction of a string literal.


In [ ]:
g = StarLayerGraph()
g.bind("ex", EX)

#literals can now include language direction.
g.add((EX.title, EX.value, DirLangString("مرحبا", "ar", "rtl")))
g.add((EX.title, EX.value, Literal("hello","en")))
g.add((EX.title, EX.value, DirLangString("hello","en","ltr")))

print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:title ex:value "hello"@en, "hello"@en--ltr, "مرحبا"@ar--rtl .



## 2. Parsing and Serialization
Supports parsing and serialization of RDF 1.2 content across the full set of rdflib supported formats. (turtle12 and longturtle12 for Turtle, nt12 and nq12 for N-Triples and N-Quads, trig12 and trix12 for datasets, rdfxml12,  and jsonld12. (Note that jsonld does not have a published RDF 1.2 spec and this serialization should only be used within starlight.)
- Quoted triple-term content
- Turtle annotation syntax
- Language-direction literals

In [ ]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
# rdflib g.parse exteded to handle RDF 1.2 terms for all rdflib supported formats.
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals
    ex:note_en ex:text "hello"@en--ltr .
    ex:note_ar ex:text "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:source ex:wikipedia ;
      ex:confidence "high" .

    # anonymous inline annotation block
    ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} .

    # named reifier with annotations
    ex:bob ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} .

    # named reifier without annotation block
    ex:bob ex:worksWith ex:frank ~ ex:stmt2 .

    # an additional  quoted triple term reused in querie examples.
    ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .
''', format='turtle12')

#rdflib g.serialize extended to handle RDF 1.2 terms.
print(g_parsed.serialize(format='turtle12'))
#other options:
#turtle12, longturtle12, nt12, nq12, trig12, trix12, rdfxml12,jsonld12

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .

ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} ;
    ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} ;
    ex:worksWith ex:frank ~ ex:stmt2 .

ex:claim ex:confidence "high" ;
    ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

ex:note_ar ex:text "مرحبا"@ar--rtl .

ex:note_en ex:text "hello"@en--ltr .



## 3. SPARQL and query semantics
Supports SPARQL 1.2 query execution.
- Query over reified quoted triples
- Allows Turtle 1.2 quoted-triple syntax in queries (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

Following examples use the graph created in section 2 above.

In [29]:
# SPARQL query to bind all terms inside a quoted triple 
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:knows)
}
ORDER BY ?claim ?s ?o
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.s),
        g_parsed.qname(row.p),
        g_parsed.qname(row.o),
        "Source: ",g_parsed.qname(row.source),
    )

ex:claim ex:bob ex:knows ex:carol Source:  ex:wikipedia


In [30]:
# detect triple-term values using isTRIPLE
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?p WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

for row in rows:
    print(g_parsed.qname(row.claim),g_parsed.qname(row.p))


ex:alice ex:mentions
ex:claim rdf:reifies
ex:stmt1 rdf:reifies
ex:stmt2 rdf:reifies
rr:0 rdf:reifies


### Additional SPARQL 1.2 functions
- `TRIPLE(s, p, o)` — a fucntional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term
- `LANGDIR()`, `hasLANGDIR()`, `STRLANGDIR()` — read, check, and build a literal's base direction
- `LANG()`, `hasLANG()` — SPARQL 1.1 functions, now also aware of direction-tagged literals

In [12]:
# TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:bob, ex:knows, ex:carol))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname(row.s), g_parsed.qname(row.p), g_parsed.qname(row.o))

ex:bob ex:knows ex:carol


In [13]:
# LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over the direction-tagged notes
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:text ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s
""")
for row in rows:
    print(g_parsed.qname(row.s), row.lang, row.dir, row.hasDir)

# STRLANGDIR() constructs a direction-tagged literal directly from plain strings
rows = g_parsed.query('SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit.n3())

ex:note_ar ar rtl true
ex:note_en en ltr true
"hi"@en--ltr


### Turtle annotation shorthand inside SPARQL queries
The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` turtle syntax used to parse and serialze graphs, also work directly inside a SPARQL 1.2 WHERE clause.  Allows quering using the shorthand the same way turtle is serialized without expanding to `rdf:reifies`/`<<( )>>`.

In [14]:
# {| ?pred ?val |}: query an anonymous reifier's annotations inline
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  ex:bob ex:likes ex:dana {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

# ~ ?r: bind the reifier itself for a named reifier
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?r WHERE {
  ex:bob ex:worksWith ex:frank ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname(row.r))

# << s p o >> ?pred ?val: reification shorthand, no assertion required
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  << ex:bob ex:likes ex:dana >> ?pred ?val
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

ex:since 2020
ex:source http://example.org/LinkedIn
ex:stmt2
ex:since 2020
ex:source http://example.org/LinkedIn


## 4. SHACL validation and rules

StarLayer-shacl extends pySHACL in line with the SHACL 1.2 draft specifications.
- Validation over RDF 1.2 graphs
- SHACL rule support
- Direction-aware datatype constraints
- Fine-grained, per-constraint severity and messages
- New severity levels (`sh:Debug`, `sh:Trace`)
- Node expressions (`shnex:`, plus `sparql:`-namespaced SPARQL built-ins)

**Not covered yet, but planned** — the project tracks this explicitly against the live W3C drafts in `packages/shacl/docs/shacl12-gap-matrix.md`, with a concrete implementation design for four of these in `packages/shacl/docs/implementation-plan.md`'s "2026-08-26 Gap-Closure Plan" (last re-checked 2026-08-26); highlights:
- `sh:RuleSet`/`sh:hasRule`/`sh:includesRuleSet` — new as of `WD-shacl12-sparql-20260821` (2026-08-21), letting a shapes graph name and select a subset of its rules; **planned**, `apply_rules()` design sketched in the implementation plan above. `apply_rules()` today always runs every rule in the graph, which happens to match the spec's *default* rule set, so nothing is incorrect yet — there's just no way to pick a non-default named one
- `sh:resultAnnotation`/`sh:annotationProperty`/`sh:annotationVarName`/`sh:annotationValue` — unimplemented in pySHACL itself, but **planned** here via a targeted patch (a real hook already exists in `SPARQLBasedConstraint.evaluate()`'s `bound_vars`)
- Content-level validation of `sh:select`/`sh:ask`/`sh:construct` query *text* — checked structurally today (cardinality, datatype), not parsed as real SPARQL; **planned** as a Python-level preflight parse
- SHACL 1.2 UI's `shui:WidgetScore`/`shui:WidgetAcceptMatcher` widget-selection algorithm — `shui:` annotations on shapes validate fine; the selection procedure itself is **planned**, pending one prerequisite re-check of the still-unstable `shui:` scoring vocabulary
- **Deferred**: SPARQL Rule Language (SRL, renamed from "Shape Rules Language" 2026-08-19, now published standalone at `sparql12-rl` outside the `shacl12-` family) and SHACL's own Compact Syntax — RDF is the interchange form both compile *to*, not a text form this library parses
- **Deferred**: `sh:PropertyRule` (Core's `sh:values`-based rule shorthand) — a genuine gap in pySHACL itself, not just untested here; using it raises `RuleLoadError`
- **Deferred**: SHACL 1.2 Profiling — investigated directly against the live spec text (re-confirmed 2026-08-26); no validator behavior gap found. Its packaging conventions (`sh:ShapesGraph`/`sh:DataGraph`, `owl:imports`) are SHOULD-level metadata with no validation effect, and the one MAY-level runtime item (the `sh:conformsTo` inference rule) is mechanically just an ordinary `sh:SPARQLRule` this engine already executes — it only needs a caller to mint real IRI identity for their data/shapes graphs first, which is an application decision, not something this library can supply a default for

### Is a value a triple term? `sh:nodeKind ( sh:TripleTerm )`
The list-valued form of `sh:nodeKind` recognizes `sh:TripleTerm` as one of its choices, letting a shape require that a value — e.g. the object of `rdf:reifies` — is genuinely an RDF 1.2 triple term, not a plain URI, blank node, or literal. Only the list form works; a bare `sh:nodeKind sh:TripleTerm` (no list) silently falls through to plain pySHACL's own check, which has no concept of triple terms at all.

In [15]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    [] a sh:NodeShape ;
      sh:targetSubjectsOf rdf:reifies ;
      sh:property [ sh:path rdf:reifies ; sh:nodeKind ( sh:TripleTerm ) ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle12")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms (real triple term):", result.conforms)

bad_data = StarLayerGraph()
bad_data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies ex:not_a_triple_term .
""", format="turtle")
result_bad = StarShaclValidator().validate(data_graph=bad_data, shacl_graph=shapes, meta_shacl=False)
print("conforms (plain URI, not a triple term):", result_bad.conforms)

conforms (real triple term): True
conforms (plain URI, not a triple term): False


### New severity levels: `sh:Debug`, `sh:Trace`
SHACL 1.2 adds two severities below `sh:Info`/`sh:Warning`. Unlike those two (which need `allow_warnings=True`/`allow_infos=True` to stop blocking `conforms`), a `sh:Debug`/`sh:Trace` result **never** blocks conformance — it's recorded in the report but has no effect on `result.conforms`, useful for constraints you want visibility into without failing validation over.

In [16]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:severity sh:Debug ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print("conforms despite missing ex:age:", result.conforms)  # True - sh:Debug never blocks
print(result.report_text)

conforms despite missing ex:age: True
Validation Report
Conforms: False
Results (1):
Validation Result in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Debug
	Source Shape: [ sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:age ; sh:severity sh:Debug ]
	Focus Node: ex:alice
	Result Path: ex:age
	Message: Less than 1 values on ex:alice->ex:age



### Fine-grained severity via reification: `{| sh:severity ... |}`
A single constraint-value triple can carry its own `sh:severity`/`sh:deactivated` override via RDF 1.2's inline annotation shorthand — distinct from, and finer-grained than, a shape's own `sh:severity`. Currently wired for `sh:datatype`/`sh:uniqueMembers`/`sh:reificationRequired`/`sh:singleLine` (severity) and `sh:property` (deactivation), and only on a constraint declared directly on the targeted shape (not yet inside a nested `sh:property [...]` blank node).

In [17]:
SH = Namespace("http://www.w3.org/ns/shacl#")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:AgeMustBeIntShape a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:datatype xsd:integer {| sh:severity sh:Warning |} .
""", format="turtle12")
data = StarLayerGraph()
data.add((EX.alice, EX.dummy, EX.alice))  # ex:alice is a URI, not an xsd:integer literal - violates

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
severities = {result.report_graph.qname(o) for _, _, o in
              result.report_graph.triples((None, SH.resultSeverity, None))}
print("severity from the annotation:", severities)  # {'sh:Warning'}, not the default sh:Violation
print("conforms:", result.conforms)                  # still False - Warning still blocks unless allowed

result2 = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False, allow_warnings=True)
print("conforms with allow_warnings=True:", result2.conforms)

severity from the annotation: {'sh:Warning'}
conforms: False
conforms with allow_warnings=True: True


In [18]:
#SHACL validation of a direction-tagged literal.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:n1 a ex:Note ;
      ex:label "hello"@en--ltr .
""", format="turtle12")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:LangShape a sh:NodeShape ;
      sh:targetClass ex:Note ;
      sh:property [ sh:path ex:label ; sh:datatype rdf:dirLangString ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print(result.conforms)

True


In [19]:
#NOTE - can we make the output clear by outputting the new triple.

#SHACL validation targets a reifier node (sh:targetSubjectsOf ex:confidence)
# and conditionally infers a new triple from its annotation - SHACL rules operating on RDF 1.2
# reification structure.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:confidence "high" .
""", format="turtle12")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ConfidenceRule a sh:NodeShape ;
    #nodes with high confidence should be trusted.
      sh:targetSubjectsOf ex:confidence ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject sh:this ;
        sh:predicate ex:trusted ;
        sh:object ex:yes ;
        sh:condition [ sh:property [ sh:path ex:confidence ; sh:hasValue "high" ] ] ;
      ] .
""", format="turtle")
result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print((EX.claim, EX.trusted, EX.yes) in result.data_graph)
print(result.conforms)

True
True


## 5. SPARQL graph, Ontology and Shapes

Starlayer-sparql represents SPARQL queries as an RDF graph based on SPARQL 1.2 algebra. These graphs use the `salg:` ontology, which is specific to Starayer. RDF-based SPARQL queries can be exported and imported. Starayer also uses a corresponding SHACL shapes graph to validate SPARQL queries.

Similarly, Starlayer-shacl extends the SHACL meta-shape graph for SHACL 1.2 semantics.

The following examples show how to import/export SPARQL queries as RDF, and how to obtain the ontology (SPARQL only) and shape  graphs (SPARQL and SHACL).   

### Importing and exporting a SPARQL query as RDF
Grounded in one tiny graph and one tiny query throughout, so each step's effect shows up as a real, different query result — not just a triple count. Six steps: build a graph + query → parse the query into a `Query` object → encode its algebra as RDF → edit the query by editing the RDF → decode back into a real `Query` and run it → export the edited query back to text.

#### Step 1 — a graph and a query
Everything below is grounded in this one tiny graph and one tiny query, so later steps' effects show up as real, different query results — not just triple counts.

In [20]:
import starsparql

g = StarLayerGraph()
g.bind("ex", EX)
g.add((EX.bob, EX.knows, EX.carol))
g.add((EX.bob, EX.likes, EX.dana))

text_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:knows ?o }"""

for row in g.query(text_query):
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:carol


#### Step 2 — parse the query text into a `Query` object
`prepare_query_12()` parses text into a real, executable-shaped `rdflib.Query` — the SPARQL-1.2-aware counterpart to plain rdflib's own `prepareQuery()`. Its grammar is a strict superset of SPARQL 1.1, so it's the right function to call here even though this particular query uses no RDF 1.2 syntax at all.

In [21]:
parsed = starsparql.prepare_query_12(text_query)

print(type(parsed))
print("top-level algebra op:", parsed.algebra.name)

<class 'rdflib.plugins.sparql.sparql.Query'>
top-level algebra op: SelectQuery


#### Step 3 — encode the algebra as RDF
This is the actual "get access to the RDF" step. `query_to_rdf()` walks `parsed.algebra` and encodes it into an `rdflib.Graph` using the `salg:` ontology — one triple pattern, filter, projection, etc. per algebra node. `root` is the graph node standing in for the query's top-level operator, needed to decode the graph back into a `Query` later.

In [22]:
graph, root = starsparql.query_to_rdf(parsed)
print("encoded triples:", len(graph))

encoded triples: 143


#### Step 4 — edit the query by editing the RDF
Rewrite the triple pattern's predicate directly in the RDF graph — `ex:knows` → `ex:likes`. No text-level find/replace anywhere; this edits the query's own structure.

In [23]:
from starsparql.vocab import SALG

for s, p, o in list(graph.triples((None, SALG.predicate, EX.knows))):
    graph.remove((s, p, o))
    graph.add((s, p, EX.likes))

#### Step 5 — decode back into a real `Query`, and run it
`rdf_to_query()` turns the edited RDF back into an executable `Query`. Running it against the *same* graph from Step 1 proves the edit is semantic — the result set genuinely changes, from `(bob, carol)` to `(bob, dana)`.

In [24]:
modified_query = starsparql.rdf_to_query(graph, root)

for row in g.query(modified_query):
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:dana


#### Step 6 — export the modified query back to text
`translate_algebra_12()` walks a `Query`'s `.algebra` straight to SPARQL 1.2 text — no intermediate RDF round-trip needed when text is all you want.

In [25]:
print(starsparql.translate_algebra_12(modified_query))

SELECT ?s ?o{?s <http://example.org/likes> ?o. }


#### Step 7 — load the ontology and shapes graphs
`starsparql.ontology_graph()` and `starsparql.shapes_graph()` each return a fresh **plain `rdflib.Graph`**, not a `StarLayerGraph` — confirmed from source (`starsparql/ontology/__init__.py`, `starsparql/ontology/sparql_shapes.py`). They contain the `salg:`/SHACL *vocabulary itself* (classes, properties, shape definitions) parsed with plain `format="turtle"`, never actual triple-term or reification data, so there's nothing RDF-1.2-specific for them to carry — a plain `rdflib.Graph` is the right and complete return type here. `shapes_graph()` additionally requires `pyshacl` to be installed.

In [26]:
ontology = starsparql.ontology_graph()
print("ontology triples:", len(ontology))

shapes = starsparql.shapes_graph()
print("shapes triples:", len(shapes))

ontology triples: 333
shapes triples: 1265


#### Step 8 — validate an encoded query, with RDFS reasoning over the ontology
`starsparql.validate()` runs the shapes graph against a data graph with `inference="rdfs"` and `ont_graph=ontology_graph()` — real RDFS reasoning before SHACL validation, which is what lets shapes like `GraphPatternShape`/`ExpressionShape` check a single abstract superclass (`salg:Expression`) instead of enumerating every concrete operator/builtin by name. This example builds its query with plain rdflib's own `prepareQuery()` (not `starsparql.prepare_query_12()`) deliberately, to show `validate()` works on any correctly-encoded `salg:` graph, regardless of what produced it.

In [27]:
from rdflib.plugins.sparql import prepareQuery

q = prepareQuery("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX})
encoded, root = starsparql.query_to_rdf(q)
print("encoded query triples:", len(encoded))

conforms, results_graph, results_text = starsparql.validate(encoded)
print("conforms:", conforms)

encoded query triples: 143
conforms: True


## 6. Backend graph-store and format support

This section is about *where the data actually lives* — which store StarLayer talks to, and whether it speaks RDF 1.1 or native RDF 1.2 to get there. (RDF 1.2 format support — turtle12, nt12, nq12, trig12, trix12, rdfxml12, jsonld12, longturtle12 — is covered separately below, in "All eight RDF 1.2 formats".)

- In-memory backend (default) and native RDF 1.2 backend modes
- Dual-mode operation: the same store can be driven in RDF 1.1 (encoding, rewritten queries) or native RDF 1.2 mode
- Any rdflib `Store` plugin works transparently under the default RDF 1.1 backend — not just the built-in in-memory store

### Oxigraph — native RDF 1.2

Oxigraph 0.5.9+ speaks SPARQL 1.2 (triple-term syntax) directly over HTTP. Point `StarLayerGraph` at it with `backend='rdf-1.2'` and a `SPARQLUpdateStore`, and triple terms/direction-tagged literals go over the wire in their real syntax — no `tt:HASH` encoding, no query rewriting. **Illustrative only** — this needs a running Oxigraph instance (`docker run -d -p 7878:7878 ghcr.io/oxigraph/oxigraph serve --location /data --bind 0.0.0.0:7878`); not executed in this notebook.

```python
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

store = SPARQLUpdateStore(
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)
g = StarLayerGraph(store=store, identifier=EX.main, backend="rdf-1.2")
g.bind("ex", EX)
g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))

rows = g.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:bob ex:knows ex:carol )>> }
""")
for row in rows:
    print(g.qname(row.stmt))
```

### Fuseki — dual-mode: RDF 1.1 and RDF 1.2 against the same store

Fuseki 5.5+ also speaks native RDF 1.2. The dual-mode story: the *same* store connection works in either backend mode — omit `backend=` (defaults to `'rdf-1.1'`) to have StarLayer encode triple terms and rewrite SPARQL 1.2 syntax down to plain SPARQL 1.1 before sending it, for compatibility with any SPARQL 1.1-only endpoint; pass `backend='rdf-1.2'` to send native RDF 1.2 syntax directly once you know the endpoint supports it. **Illustrative only** — needs a running Fuseki instance (`docker run -d -p 3030:3030 atomgraph/fuseki:latest --update --mem --ping /starlayergraph`, Fuseki 5.5+ required for the RDF 1.2 mode); not executed here.

```python
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

def make_store():
    return SPARQLUpdateStore(
        query_endpoint="http://localhost:3030/starlayergraph/query",
        update_endpoint="http://localhost:3030/starlayergraph/update",
        auth=("admin", "admin"),
    )

# RDF 1.1 mode: triple terms encoded, SPARQL 1.2 rewritten to 1.1 before sending
g11 = StarLayerGraph(store=make_store(), identifier=EX.main)   # backend='rdf-1.1' is the default
g11.bind("ex", EX)
g11.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))

# RDF 1.2 mode: native triple-term syntax sent directly, no rewriting
g12 = StarLayerGraph(store=make_store(), identifier=EX.main, backend="rdf-1.2")
g12.bind("ex", EX)
rows = g12.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:bob ex:knows ex:carol )>> }
""")
for row in rows:
    print(g12.qname(row.stmt))
```

### SQLAlchemy — RDF 1.1, real SQL-backed persistence

Unlike Oxigraph/Fuseki above, this one *is* executed in this notebook: `StarLayerGraph` is an ordinary `rdflib.Graph` subclass, so any rdflib `Store` plugin works transparently under the default RDF 1.1 (encoding) backend — including a real SQL database via `rdflib-sqlalchemy`. Needs the `sqlalchemy` extra (`pip install rdflib-sqlalchemy`); no native RDF 1.2 mode exists for this store (`rdflib-sqlalchemy` has no SPARQL 1.2 support of its own), so this is RDF 1.1-only.

In [28]:
!pip install -q rdflib-sqlalchemy

import tempfile
import rdflib_sqlalchemy
rdflib_sqlalchemy.registerplugins()

db_path = tempfile.mktemp(suffix=".sqlite")
uri = f"sqlite:///{db_path}"

writer = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
writer.open(uri, create=True)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.commit()
writer.close()

# fresh graph object, same database file - proves the data actually persisted
reader = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
reader.open(uri, create=False)
reader.bind("ex", EX)
print(reader.qname_term(next(reader.triple_terms(subject=EX.bob))))
reader.close()

<<( ex:bob ex:knows ex:carol )>>


/Users/johnclements/Documents/GitHub/hidden-graph/starlayer/.venv/lib/python3.14/site-packages/rdflib_sqlalchemy/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution
